# ② 검색 — 색인 만들기 · 검색
담당: 검색 · 코드: `rag/retrieval/indexing.py`, `rag/retrieval/retrieval.py`

01 전처리의 `artifacts/chunks.jsonl` 로 Dense(KURE-v1 임베딩 → FAISS)와 BM25(Kiwi 형태소) 색인을 만들어 `artifacts/index/` 에 저장합니다.
- 임베딩은 청크를 `embedding.max_seq_length`(2,048) 토큰까지 읽습니다 — 표 청크가 잘리지 않습니다.
- BM25 는 요구사항 코드(`SFR-001`)를 한 토큰으로, 부정어 '않' · '아니' 도 남기고, 사용자 사전 · 별칭(부가세 → 부가가치세, 지체 상금 → 지체상금)으로 질문과 문서의 낱말을 맞춥니다.

검색 전에 OpenAI API로 질문을 재작성합니다(유료). 재작성한 질문으로 Dense·BM25 검색을 하고, 재작성한 질문에 나온 발주기관 문서 안에서만 찾습니다 (후속 질문의 '그 사업'도 재작성 때 사업명으로 풀립니다).
색인은 청크의 **검색 글**로 만듭니다. 요구사항에 속한 청크의 글에 그 요구사항 ID나 명칭이 없으면 `[요구사항 SER-001 보안관리 및 규정 준수]`를 앞에 붙여,
ID · 명칭 청크와 나뉜 상세 내용 청크도 요구사항 ID · 이름으로 찾히게 합니다(전처리 패키지의 `requirement_key`). 답변 근거로는 원래 글을 그대로 넘깁니다.
검색 글 만드는 방식이 바뀌었으므로 **기존 색인을 다시 만드세요** — 색인이 지금 청크와 맞지 않으면 검색할 때 다시 만들라고 알려 줍니다.

검색은 Dense · BM25 하이브리드이며, 두 검색의 후보(각각 상위 `candidate_k`)를 **RRF**(Reciprocal Rank Fusion)로 합칩니다 — 후보마다 `Σ 1/(60 + 순위)`.
점수 크기가 아니라 각 검색에서의 순위만 써서, 척도가 다른 코사인 · BM25 점수를 비율(예전 `dense_weight`)로 섞을 때처럼 질문마다 뜻이 달라지지 않고 정할 비율도 없습니다. 두 검색 모두에서 위에 있는 청크가 앞에 옵니다.
- 질문의 요구사항 코드가 그대로 든 청크를 맨 앞에 둡니다. 전처리 패키지를 쓰면 본문에 코드가 없어도 그 요구사항에 속한 청크(상세 내용)도 함께 맨 앞에 둡니다.
- 찾은 청크에 연결 청크가 있으면 바로 뒤에 함께 담습니다 — 같은 요구사항의 다른 청크(ID · 명칭 / 상세 내용, `requirement_key`)와 여러 청크로 나뉜 같은 표의 다른 부분(`table_refs`, 6청크 이하 표).
  찾은 청크 하나에 붙이는 연결 청크는 600토큰까지라 연결 청크가 토큰 예산을 다 차지하지 않습니다.
- 질문에 기관이 둘 이상이면(비교 질문) 기관마다 따로 찾고 토큰 예산도 기관마다 따로 씁니다. 같은 문서를 가리키는 기관명은 하나로 합치고, 질문에 사업명이 나오면 같은 기관의 다른 사업 문서는 뒤로 미룹니다.
- 질문에 발주기관이 나오면 그 기관 문서 안에서만 찾습니다. 기관명은 발주 기관 목록 CSV(`paths.org_csv`)와 문서 표지에서 모읍니다.
- 기관 문서 안에서는 질문에서 기관 · 사업을 가리킨 부분(`스포츠윤리센터의 「… LMS 기능개선」`)을 빼고 순위를 매깁니다. 범위를 이미 그 문서로 좁혔으니
  이 낱말들은 청크를 가르지 못하고, 사업명을 되풀이하는 표지 · 서식 청크만 앞세워 묻는 내용(`하자보증 기간`)의 청크를 예산 밖으로 밀어냈습니다.
  CSV는 증강 Golden dev 396 패키지의 `data/data_list.csv`(UTF-8, `파일명` · `발주기관` 열)를 그대로 씁니다.
- RRF 상위 후보(`rerank.top_n`)는 로컬 리랭커(크로스 인코더, `rerank.model`)가 질문과 청크를 함께 읽어 순서를 다시 매깁니다. GPU에서 돌아 API 비용이 없습니다.
- 결과는 개수가 아니라 토큰 예산(`retrieval.token_budget`)으로 자르고, 같은 문서에서 숫자 · 부정어까지 같은 거의 같은 청크만 하나로 줄입니다 (`2초/5초`, `포함한다/포함하지 않는다` 는 둘 다 남김).

`.env` 에 Langfuse 키가 있으면 검색할 때마다 Langfuse Cloud 에 `retrieve-context` 로 기록됩니다 (없어도 똑같이 동작).

In [ ]:
import pandas as pd

from rag.config import ARTIFACTS, load_config, read_jsonl, write_jsonl
from rag.retrieval.indexing import build_index
from rag.retrieval.retrieval import HybridIndex
from rag.retrieval.rewrite import rewrite_question
from rag.usage import usage_summary

cfg = load_config()

## 설정
초기값은 `config.yaml`에서 읽습니다. 아래 변수의 오른쪽을 원하는 값으로 바꾸고 설정 셀부터 다시 실행하세요.
`RERANK = False`이면 리랭커 없이 RRF 순서 그대로 담습니다. 켜면 처음 실행할 때 리랭커 모델(약 2.3GB)을 내려받습니다.
적용한 rewrite·검색 설정을 `artifacts/retrieval_settings.jsonl`에 저장합니다. 04는 이 파일을 읽어 같은 설정으로 검색하므로, 값을 바꾼 뒤에는 04도 다시 실행하세요.

In [ ]:
REWRITE_MODEL = "gpt-6-luna"               # 예: "gpt-5-mini"
REWRITE_REASONING_EFFORT = "low"                        # 예: "low"
REWRITE_MAX_TOKENS = cfg["rewrite"]["max_tokens"]             # 출력·추론 토큰 한도
REWRITE_PRICES = cfg["rewrite"]["usd_per_1m_tokens"]          # 입력 · 캐시 입력 · 출력 단가
RERANK = True                           # [True 기본 ]로컬 리랭커 사용 (True / False)

cfg["rewrite"].update(model=REWRITE_MODEL, reasoning_effort=REWRITE_REASONING_EFFORT,
                      max_tokens=REWRITE_MAX_TOKENS, usd_per_1m_tokens=REWRITE_PRICES)
cfg["rerank"]["enabled"] = RERANK
write_jsonl([{"rewrite": cfg["rewrite"], "retrieval": cfg["retrieval"], "rerank": cfg["rerank"]}],
            ARTIFACTS / "retrieval_settings.jsonl")
cfg["rewrite"], cfg["rerank"]

## 1. 색인 만들기
임베딩을 청크 1,000개씩 묶어 `artifacts/index/checkpoint/` 에 저장합니다. 중간에 끊겨도 이 셀을 다시 실행하면 저장된 묶음은 건너뛰고(`저장된 결과`) 남은 묶음부터 이어서 만듭니다.
01을 다시 돌려 청크가 바뀌거나 `embedding` 설정을 바꾸면 내용이 달라진 묶음은 새로 계산합니다 (앞쪽 청크가 바뀌면 뒤 묶음도 밀려 대부분 새로 계산). BM25 토큰은 금방 끝나 매번 새로 만듭니다.
`임베딩에서 잘린 청크` 가 0 이면 모든 청크가 끝까지 임베딩된 것입니다.
`요구사항 머리말을 붙인 청크`는 검색 글 앞에 요구사항 ID · 명칭을 붙인 청크 수입니다(패키지 코퍼스는 822개). 색인과 함께 검색 글 해시(`texts.sha1`)를 저장합니다.

In [ ]:
chunks = read_jsonl(ARTIFACTS / "chunks.jsonl")
build_index(chunks, cfg, ARTIFACTS / "index")

## 2. 검색해 보기
위 설정 셀에서 rewrite·리랭커를 조절한 결과를 확인하세요. `score` 는 RRF 점수(순위 역수 합)이고, 리랭커를 켜면 리랭커 관련도(0~1)입니다. 결과 길이는 `config.yaml` 의 `retrieval` (token_budget · candidate_k)이 정합니다.

In [ ]:
index = HybridIndex(ARTIFACTS, cfg)
QUESTION = "고려대학교 차세대 포털 시스템 사업의 응답 시간 요구사항은?"
rewrite = rewrite_question(QUESTION, cfg["rewrite"])
print(rewrite)
pd.DataFrame(index.search(QUESTION, query=rewrite["question"]))[["score", "title", "page", "kind", "section", "body"]]

## 3. 기관 라우팅 점검
기관명이 하나도 연결되지 않은 문서입니다. 이 문서들은 질문에 기관명이 나와도 전체 문서에서 찾습니다.

In [ ]:
unrouted = index.router.unrouted_titles()
print(f"라우팅되는 문서 {len(index.router.titles) - len(unrouted)}/{len(index.router.titles)}")
pd.DataFrame({"미라우팅 문서": unrouted})

## API 사용량
02와 04의 rewrite 호출은 모두 `artifacts/api_usage.csv`에 자동 추가됩니다. 아래 표는 누적 사용량입니다.

In [ ]:
usage_summary("rewrite")